In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch
from gluonts.dataset.util import to_pandas

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.dataset import GiftEvalDataset
from tsfm_lens.metrics import compute_metrics
from tsfm_lens.utils import (
    apply_custom_style,
    clear_cuda_cache,
    load_dyst_samples,
    set_seed,
)

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
clear_cuda_cache(device)
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../../figs", "ablations_chronos", "examples")
os.makedirs(plot_save_dir, exist_ok=True)


### Load Pipeline

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.num_layers
num_heads = pipeline.num_heads
print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
pipeline.__class__.__name__

### Load Data

In [ ]:
split_name = "gift-eval"

subsample_interval = 1

if split_name == "base40":
    split_dir = os.path.join(DATA_DIR, split_name)
    system_name = "Lorenz"

    context_length = 512
    context_start_time = 2000
    context_end_time = context_start_time + context_length

    # Prepare input series
    sample_idx = 0
    selected_dim = 0
    dyst_coords = load_dyst_samples(system_name, split_dir, one_dim_target=False, num_samples=1)
    dyst_coords = torch.tensor(dyst_coords[sample_idx, selected_dim, :]).unsqueeze(0)
    dyst_coords = dyst_coords[:, ::subsample_interval]
    context = dyst_coords[:, context_start_time:context_end_time]

    print(context.shape)

    prediction_length = 512
    future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
    print(f"future_vals shape: {future_vals.shape}")

elif split_name == "gift-eval":
    # Load the path to gift-eval data, which is stored in .env file
    split_dir = os.path.join(DATA_DIR, split_name)
    # system_name = "electricity/D"
    # system_name = "m4_monthly"
    system_name = "hospital"

    to_univariate = False
    term = "short"

    dataset = GiftEvalDataset(name=system_name, term=term, to_univariate=to_univariate, data_dir=split_dir)
    print(f"Prediction length: {dataset.test_data.prediction_length}")
    print(f"Number of windows: {dataset.windows}")
    print(dataset.test_data.windows)
    print(f"Number of rows in the dataset: {dataset.hf_dataset.num_rows}")

    # test_data_iter = iter(dataset.test_data)
    # test_data = next(test_data_iter)
    context_iter = iter(dataset.test_data.input)
    future_iter = iter(dataset.test_data.label)
    selected_sample_idx = 0
    for _ in range(selected_sample_idx + 1):
        context_entry = next(context_iter)
        future_entry = next(future_iter)

    context_target = context_entry["target"]
    future_target = future_entry["target"]
    print(f"context target shape: {context_target.shape}")
    print(f"future target shape: {future_target.shape}")

    fig, ax = plt.subplots(figsize=(5, 2))
    context = to_pandas(context_entry)
    future_vals = to_pandas(future_entry)
    print(f"context shape: {context.shape}")
    print(f"future shape: {future_vals.shape}")
    # prediction_length = future_vals.shape[0]
    prediction_length = 512
    print(f"prediction length: {prediction_length}")
    context.plot(color="black", linewidth=1, ax=ax)
    future_vals.plot(color="tab:blue", linewidth=1, ax=ax)
    ax.grid(which="both")
    # ax.legend("ground truth", loc="upper left")
    plt.show()

    context = torch.tensor(context)
    future_vals = torch.tensor(future_vals)

### Ablations

In [ ]:
pipeline.remove_all_hooks()

# layers_to_ablate = []
# layers_to_ablate = [0]
# layers_to_ablate = [7, 8, 9, 10]
# layers_to_ablate = [6, 7, 8, 9, 10]
# layers_to_ablate = [6, 7, 8, 9]
# layers_to_ablate = [5, 6, 7, 8]
# layers_to_ablate = [4, 5, 6, 7]
# layers_to_ablate = [3, 4, 5, 6]
# layers_to_ablate = [2, 3, 4, 5]

# heads_to_ablate = [(layer, head) for layer in layers_to_ablate for head in range(num_heads)]

# # NOTE: this is the top 20% of highest entropy heads (iirc). Note that it preserves the parroting
# # Caveat: sometimes I play with not ablating heads in some layers that are heavily represented here - on certain examples that does better
# heads_to_ablate = [
#     # (0, 4),
#     (9, 5),
#     # (10, 11),
#     (10, 5),
#     (0, 11),
#     (5, 8),
#     (10, 4),
#     (10, 8),
#     (8, 6),
#     (10, 1),
#     (5, 2),
#     (7, 2),
#     (10, 10),
#     (9, 1),
#     (2, 8),
#     # (0, 1),
#     (9, 0),
#     # (0, 10),
#     (10, 9),
#     (10, 7),
#     (1, 1),
#     (6, 10),
#     (2, 0),
#     (8, 3),
#     (10, 0),
#     (5, 1),
#     (9, 4),
#     # (0, 6),
#     (2, 2),
#     (11, 6),
#     (4, 6),
#     (2, 4),
#     (7, 10),
#     (4, 3),
#     # (0, 7),
#     # (0, 5),
# ]

# NOTE: this is the top 30% of highest entropy heads (iirc). Note that it preserves the parroting
heads_to_ablate = [
    # (0, 4),
    (10, 11),
    (9, 5),
    (10, 5),
    # (0, 11),
    (5, 8),
    (8, 6),
    (10, 8),
    (5, 2),
    (10, 1),
    (10, 4),
    (7, 2),
    (10, 10),
    (9, 1),
    (0, 10),
    (2, 8),
    (10, 9),
    (9, 0),
    # (0, 1),
    (10, 7),
    (1, 1),
    (6, 10),
    (10, 0),
    (2, 0),
    (11, 6),
    (8, 3),
    (9, 4),
    (2, 2),
    (5, 1),
    # (0, 6),
    (4, 6),
    (2, 4),
    (0, 7),
    (7, 10),
    (9, 3),
    (4, 3),
    (6, 2),
    (11, 5),
    # (0, 5),
    (7, 1),
    (7, 3),
    (6, 6),
    (3, 7),
    (6, 8),
    # (0, 0),
    (11, 11),
    (7, 8),
    (10, 2),
]

# # NOTE: this is top 50% of highest entropy heads. See how this seems to amplify the parroting
# heads_to_ablate = [
#     # (0, 4),
#     (10, 11),
#     (9, 5),
#     (10, 5),
#     # (0, 11),
#     (5, 8),
#     (8, 6),
#     (10, 8),
#     (5, 2),
#     (10, 1),
#     (10, 4),
#     (7, 2),
#     (10, 10),
#     (9, 1),
#     # (0, 10),
#     (2, 8),
#     (10, 9),
#     (9, 0),
#     # (0, 1),
#     (10, 7),
#     (1, 1),
#     (6, 10),
#     (10, 0),
#     (2, 0),
#     (11, 6),
#     (8, 3),
#     (9, 4),
#     (2, 2),
#     (5, 1),
#     # (0, 6),
#     (4, 6),
#     (2, 4),
#     # (0, 7),
#     (7, 10),
#     (9, 3),
#     (4, 3),
#     (6, 2),
#     (11, 5),
#     # (0, 5),
#     (7, 1),
#     (7, 3),
#     (6, 6),
#     (3, 7),
#     (6, 8),
#     # (0, 0),
#     (11, 11),
#     (7, 8),
#     (10, 2),
#     (11, 3),
#     (7, 9),
#     (1, 3),
#     (6, 11),
#     # (0, 2),
#     (7, 4),
#     (8, 5),
#     (4, 0),
#     (2, 6),
#     (2, 11),
#     (5, 7),
#     (9, 2),
#     # (0, 9),
#     (11, 1),
#     (6, 1),
#     (6, 5),
#     # (0, 8),
#     (10, 6),
#     (4, 11),
#     (1, 7),
#     (11, 7),
#     (11, 2),
#     (2, 5),
#     (11, 0),
# ]

# NOTE: these are the lowest entropy heads. Ablating just a single one of these breaks the parroting
# heads_to_ablate = [
#     (4, 10),
#     # (3, 10),
#     # (4, 1),
#     # (7, 11),
#     # (6, 9),
#     # (3, 9),
#     # (3, 1),
#     # (2, 3),
#     # (5, 6),
#     # (3, 2),
#     # (5, 3),
#     # (3, 5),
#     # (1, 2),
#     # (2, 7),
#     # (6, 4),
#     # (8, 8),
#     # (5, 0),
#     # (7, 0),
#     # (5, 5),
#     # (5, 11),
#     # (1, 11),
#     # (8, 10),
#     # (2, 10),
#     # (8, 0),
#     # (11, 4),
#     # (4, 8),
#     # (8, 1),
#     # (8, 2),
#     # (8, 7),
#     # (4, 5),
#     # (3, 3),
#     # (8, 9),
#     # (3, 4),
#     # (8, 11),
#     # (0, 3),
#     # (6, 7),
# ]


######### Attention Head ablations #########
pipeline.add_head_ablation_hooks(
    heads_to_ablate,
    attention_type="ca",
)
pipeline.add_head_ablation_hooks(
    heads_to_ablate,
    attention_type="sa",
)

######### MLP ablations #########
# pipeline.add_mlp_ablation_hooks(layers_to_ablate, ablation_method="zero")
# pipeline.add_mlp_ablation_hooks([1, 2], ablation_method="zero")

In [ ]:
layers_without_ca_heads = list(pipeline.ca_head_ablation_handles.keys())
layers_without_sa_heads = list(pipeline.sa_head_ablation_handles.keys())
layers_without_mlps = list(pipeline.mlp_ablation_handles.keys())

assert layers_without_ca_heads == layers_without_sa_heads, "CA and SA ablations must be identical"
layers_without_heads = layers_without_ca_heads

if layers_without_heads and layers_without_mlps:
    if layers_without_heads != layers_without_mlps:
        # raise NotImplementedError("Zero-ablation of heads and MLPs in different layers is messier, save for later")
        pass
    ablations_summary_str_title = f"Zero-Ablation: Heads and MLPs in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_mlps_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_heads:
    ablations_summary_str_title = f"Zero-Ablation: Heads in Layers {layers_without_heads}"
    ablations_summary_str = f"za_heads_layers_{'-'.join(map(str, layers_without_heads))}"
elif layers_without_mlps:
    ablations_summary_str_title = f"Zero-Ablation: MLPs in Layers {layers_without_mlps}"
    ablations_summary_str = f"za_mlps_layers_{'-'.join(map(str, layers_without_mlps))}"
else:
    ablations_summary_str = ablations_summary_str_title = None

print(ablations_summary_str)
print(ablations_summary_str_title)


### Predict and Get Outputs

In [ ]:
rseed = 123
set_seed(rseed)

num_samples = 10
do_sample = True if num_samples > 1 else False

# pred, _ = pipeline.predict_with_full_outputs(  # type: ignore
#     context=context,
#     prediction_length=prediction_length,
#     num_samples=num_samples,
#     do_sample=do_sample,  # greedy decoding if num_samples == 1
#     use_cache=True,
#     return_dict_in_generate=True,
#     output_scores=True,
#     limit_prediction_length=False,
# )

pred = pipeline.predict(  # type: ignore
    context=context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    limit_prediction_length=False,
)

### Plot Predictions

In [ ]:
# Prepare context and predictions
context_np = context.squeeze().cpu().numpy()
context_timesteps = np.arange(context_np.shape[-1])
future_vals_np = future_vals.squeeze()
future_timesteps = np.arange(context_np.shape[-1], context_np.shape[-1] + prediction_length)
print(f"context_np shape: {context_np.shape}")
print(f"future_vals_np shape: {future_vals_np.shape}")
print(f"length of context_timesteps: {len(context_timesteps)}")
print(f"length of future_timesteps: {len(future_timesteps)}")

In [ ]:
preds = pred.squeeze()
if preds.ndim == 1:
    preds = preds[None, :]

fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps, context_np, color="black", linewidth=1, label="Context")
ax.plot(
    future_timesteps[: future_vals_np.shape[-1]],
    future_vals_np,
    color="black",
    linewidth=1,
    linestyle="--",
    label="Ground Truth",
)

for i in range(len(preds)):
    ax.plot(future_timesteps, preds[i], color=DEFAULT_COLORS[i], linewidth=1, alpha=0.5)
ax.plot(future_timesteps, np.median(preds, axis=0), color="tab:blue", linewidth=2, label="Median")

# ax.set_ylim(-20, 30)
ax.set_xlabel("Timestep", fontweight="bold")

# Save plot
save_fname = (
    f"predictions_{system_name}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name}"
)
save_path = os.path.join(plot_save_dir, "zero_ablations", f"{ablations_summary_str}_{save_fname}.pdf")

if ablations_summary_str_title:
    print(ablations_summary_str_title)
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight")

In [ ]:
pipeline.remove_all_hooks()
set_seed(rseed)

pred_original = pipeline.predict(  # type: ignore
    context=context,
    prediction_length=prediction_length,
    num_samples=num_samples,
    limit_prediction_length=False,
)

In [ ]:
preds_original = pred_original.squeeze()
if preds_original.ndim == 1:
    preds_original = preds_original[None, :]

fig, ax = plt.subplots(figsize=(6, 2))

# Plot context, ground truth and predictions
ax.plot(context_timesteps, context_np, color="black", linewidth=1, label="Context")
ax.plot(
    future_timesteps[: future_vals_np.shape[-1]],
    future_vals_np,
    color="black",
    linewidth=1,
    linestyle="--",
    label="Ground Truth",
)
for i in range(len(preds_original)):
    ax.plot(future_timesteps, preds_original[i], color=DEFAULT_COLORS[i], linewidth=1, alpha=0.5)
ax.plot(future_timesteps, np.median(preds_original, axis=0), color="tab:blue", linewidth=2, label="Median")

ax.set_xlabel("Timestep", fontweight="bold")

# Save plot
save_fname = (
    f"predictions_{system_name}_dim{selected_dim}_start{context_start_time}_sub{subsample_interval}"
    if split_name == "base40"
    else f"predictions_{system_name}"
)
save_path = os.path.join(plot_save_dir, f"{save_fname}.pdf")
print(f"Saving to {save_path}")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.tight_layout()
fig.savefig(save_path, bbox_inches="tight")

### Compute Metrics

In [ ]:
med_pred = np.median(preds, axis=0)
med_orig = np.median(preds_original, axis=0)
print(f"med_pred shape: {med_pred.shape}")
print(f"med_orig shape: {med_orig.shape}")

In [ ]:
pred_horizon_lst = [512]
for pred_horizon in pred_horizon_lst:
    print(f"Computing metrics for prediction horizon {pred_horizon}")
    metrics_combined = compute_metrics(med_orig[:pred_horizon], med_pred[:pred_horizon])
    pprint(metrics_combined, width=1, indent=2)